# AgentCore Harness — Restaurant Concierge

> **Time budget:** ~45-60 minutes. This is core **live-session** content (not pre-work).
> **Prerequisites:** notebooks 0-3 completed; `requirements.txt` installed (includes
> `bedrock-agentcore-starter-toolkit`, which provides the `agentcore` CLI used throughout
> this notebook).

## AgentCore Harness

This notebook builds a restaurant-booking scenario on **Amazon Bedrock AgentCore**, using the
**AgentCore harness**: a config-based way to build an agent — you declare a model, a system
prompt, tools, and memory, and the harness runs the request/tool-call/response loop for you,
in a managed, isolated microVM per session.

AgentCore also supports **code-defined agents on AgentCore Runtime**, where you write your own
orchestration loop using a framework (Strands, LangChain, the OpenAI Agents SDK, the Claude
Agent SDK) or custom code. This workshop uses the harness, so there's no agent loop to write —
you configure JSON. Code-defined agents are for cases the harness genuinely can't express (see
notebook 5's closing note on agent-to-agent protocols).

## Agent vs. LLM call — a 5-minute primer

A **plain LLM call** is one request, one response: you send a prompt, the model replies, done.

An **agent** wraps the model in a *loop*:

```
 ┌─────────────────────────────────────────────────────────────┐
 │                                                               │
 │   user turn ──▶ model decides: answer directly, OR            │
 │                  call a tool                                  │
 │                     │                                         │
 │                     ▼                                         │
 │              tool runs, result comes back                     │
 │                     │                                         │
 │                     ▼                                         │
 │              model sees the result, decides again ────────────┤
 │                     │                                         │
 │                     ▼ (until it has enough to answer)          │
 │                final answer to the user                       │
 │                                                               │
 └─────────────────────────────────────────────────────────────┘
```

The model itself decides *whether* to call a tool, *which* tool, and *when* it has enough
information to stop — that decision loop is what makes something "agentic" rather than a
single completion.

**Where the AgentCore harness fits:** you never write that loop. You declare:
- a **model** (which LLM answers/decides),
- a **system prompt** (persona and constraints),
- **tools** (what the model is allowed to call — inline functions or Gateway-fronted APIs/Lambdas),
- **memory** (what persists across turns and sessions),

and the harness runs the loop above for you, end to end, as a managed service. That's the
entire mental model for the rest of this notebook.

## Setup

Load credentials and create the boto3 clients we'll need to deploy the small IAM-role
CloudFormation stack and the restaurant-facts Lambda.

In [ ]:
from dotenv import load_dotenv
import json
import os
import sys
import subprocess
import time
import uuid as uuid_module

sys.path.insert(0, "resources/src")
import boto3
import cloudformation_utils
import utils

aws_region = "us-east-1"
load_dotenv(".env")

cloudformation_client = boto3.client("cloudformation", region_name=aws_region)

UUID = uuid_module.uuid4().hex[:8]
print("Session UUID suffix:", UUID)

In [ ]:
# Sanity-check that the `agentcore` CLI is on PATH (installed via requirements.txt)
result = subprocess.run(["agentcore", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

### Deploy the harness execution role

The AgentCore CLI can auto-create a default execution role for a harness if you don't supply
one -- that default role is enough for a **prompt-only** harness, but it does **not** include
the `bedrock-agentcore` Memory actions we'll need in Step 2. We deploy a small, dedicated role
up front so we only redeploy the harness (not re-create IAM) when we wire memory in.

In [ ]:
template_body = cloudformation_utils.load_template("resources/infra/agentcore_harness_role_template.yaml")

stack_name = f"agentcore-restaurant-harness-role-{UUID}"
parameters = [{"ParameterKey": "UUID", "ParameterValue": UUID}]

cloudformation_utils.create_stack(aws_region, stack_name, template_body, parameters)
cloudformation_utils.wait_for_stack(aws_region, stack_name, "CREATE_COMPLETE")

In [ ]:
stack_outputs = cloudformation_client.describe_stacks(StackName=stack_name)["Stacks"][0]["Outputs"]
HARNESS_ROLE_ARN = next(o["OutputValue"] for o in stack_outputs if o["OutputKey"] == "HarnessExecutionRoleArn")
print("Harness execution role ARN:", HARNESS_ROLE_ARN)

## Step 1 — a prompt-only harness

Every harness is defined by a `harness.json` config file. The fields we'll use in this
notebook:

| Field | Meaning |
|---|---|
| `name` | The harness's name (also used as its project directory name) |
| `model.provider` / `model.modelId` | Which model answers -- here Bedrock + `global.anthropic.claude-sonnet-4-6` |
| `tools` | List of tools the model may call (empty for now) |
| `skills` | List of Markdown skills attached to the harness (empty for now -- see notebook 6) |
| `authorizerType` | Who can invoke this harness (`AWS_IAM` for this workshop) |
| `maxIterations` / `timeoutSeconds` | Cost/runaway-loop guardrails on the tool-call loop |
| `executionRoleArn` | The IAM role the harness runs as |

We write the system prompt and `harness.json` directly from this notebook -- nothing here is
a checked-in template, so you can see and tweak every field before deploying.

In [ ]:
PROJECT_NAME = "RestaurantConciergeDemo"
HARNESS_NAME = "restaurant_concierge"
MODEL_ID = "global.anthropic.claude-sonnet-4-6"

system_prompt = """You are the concierge assistant for The Bedrock Bistro, built for the Amazon Bedrock GenAI workshop.

Help guests learn about the menu, today's specials, and opening hours, and take reservation requests.

Answer in English unless the guest asks for another language.
Be warm, concise, and precise.

If you are not sure about an operational detail (menu, hours, availability), say that you need to check the restaurant's data instead of inventing facts.
"""

print(system_prompt)

In [ ]:
# Create the AgentCore project (only needs to run once -- skip if RestaurantConciergeDemo/ already exists)
if not os.path.isdir(PROJECT_NAME):
    create_cmd = [
        "agentcore", "create",
        "--project-name", PROJECT_NAME,
        "--name", HARNESS_NAME,
        "--model-provider", "Bedrock",
        "--model-id", MODEL_ID,
        "--timeout", "120",
        "--max-iterations", "10",
        "--no-harness-memory",
        "--skip-git",
        "--skip-python-setup",
        "--skip-install",
    ]
    print("Running:", " ".join(create_cmd))
    subprocess.run(create_cmd, check=True, env={**os.environ, "AWS_REGION": aws_region, "AWS_DEFAULT_REGION": aws_region})
else:
    print(f"{PROJECT_NAME}/ already exists, skipping agentcore create")

In [ ]:
# Write the system prompt into the generated project, and prepare where harness.json will go
harness_dir = os.path.join(PROJECT_NAME, "app", HARNESS_NAME)
os.makedirs(harness_dir, exist_ok=True)

with open(os.path.join(harness_dir, "system-prompt.md"), "w") as f:
    f.write(system_prompt)

harness_json_path = os.path.join(harness_dir, "harness.json")

### Your turn: build `harness_config`

Using the field table above and the variables already defined (`HARNESS_NAME`, `MODEL_ID`,
`HARNESS_ROLE_ARN`), build the `harness_config` dict and write it to `harness_json_path`:

- `name`: `HARNESS_NAME`
- `model`: `{"provider": "bedrock", "modelId": MODEL_ID}`
- `tools`: empty list for now (Step 3 appends to it)
- `skills`: empty list for now (see notebook 6)
- `authorizerType`: `"AWS_IAM"`
- `maxIterations`: `10`, `timeoutSeconds`: `120`
- `executionRoleArn`: `HARNESS_ROLE_ARN`

<details>
<summary>Click here for the solution</summary>

```python
harness_config = {
    "name": HARNESS_NAME,
    "model": {"provider": "bedrock", "modelId": MODEL_ID},
    "tools": [],
    "skills": [],
    "authorizerType": "AWS_IAM",
    "maxIterations": 10,
    "timeoutSeconds": 120,
    "executionRoleArn": HARNESS_ROLE_ARN,
}

with open(harness_json_path, "w") as f:
    json.dump(harness_config, f, indent=2)
    f.write("\n")

print(json.dumps(harness_config, indent=2))
```

</details>

In [ ]:
deploy_env = {**os.environ, "AWS_REGION": aws_region, "AWS_DEFAULT_REGION": aws_region}
subprocess.run(["agentcore", "deploy"], cwd=PROJECT_NAME, check=True, env=deploy_env)

### Invoke the prompt-only harness

AgentCore harness session IDs must be **at least 33 characters** -- a plain `uuid4()` hex
string (32 chars) is actually *just under* the limit, so we prefix it, matching the
`uuidgen`-based pattern used throughout this notebook.

In [ ]:
SESSION_1 = f"restaurant-demo-{uuid_module.uuid4()}"
print("SESSION_1:", SESSION_1, "len:", len(SESSION_1))

invoke_cmd = [
    "agentcore", "invoke",
    "--session-id", SESSION_1,
    "--stream",
    "Hi, I'm thinking about dinner tonight. What makes The Bedrock Bistro worth a visit?",
]
subprocess.run(invoke_cmd, cwd=PROJECT_NAME, check=True, env=deploy_env)

**Point to make:** the assistant is useful, but it only has prompt behavior right now. Ask
it something operational (an exact opening time, a specific menu item) and notice it either
hedges or -- worse -- makes something up. That's exactly the gap Steps 3-4 close with tools.

## Step 2 — Memory

Memory turns the harness from a stateless chatbot into something that carries guest
preferences **across turns and even across sessions**, without you writing a single line of
storage code.

We provision a memory resource with all four strategies (semantic, user-preference,
summarization, episodic), wire its name into `harness.json`, and redeploy.

In [ ]:
memory_name = f"{HARNESS_NAME}Memory"

add_memory_cmd = [
    "agentcore", "add", "memory",
    "--name", memory_name,
    "--strategies", "SEMANTIC,USER_PREFERENCE,SUMMARIZATION,EPISODIC",
]
subprocess.run(add_memory_cmd, cwd=PROJECT_NAME, check=True, env=deploy_env)

In [ ]:
harness_config["memory"] = {"name": memory_name}
with open(harness_json_path, "w") as f:
    json.dump(harness_config, f, indent=2)
    f.write("\n")

print(json.dumps(harness_config, indent=2))

In [ ]:
subprocess.run(["agentcore", "deploy"], cwd=PROJECT_NAME, check=True, env=deploy_env)

### Demonstrate memory: same session, then a brand-new session

Memory is keyed on **`--actor-id`**, not `--session-id`. We'll show three turns:

1. Store a preference in session A.
2. Recall it immediately, still in session A (in-context).
3. Open a **brand-new session B** (different `--session-id`) with the **same `--actor-id`** and
   show the preference still comes back -- from long-term memory, not conversation history.

In [ ]:
ACTOR_ID = f"guest-{int(time.time())}"
SESSION_A = f"restaurant-demo-{uuid_module.uuid4()}"
print("ACTOR_ID:", ACTOR_ID)
print("SESSION_A:", SESSION_A)

subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_A, "--actor-id", ACTOR_ID, "--stream",
     "Remember that I'm vegetarian and prefer a window seat. Reply with just: OK."],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

In [ ]:
# Turn 2 -- same session, should recall immediately
subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_A, "--actor-id", ACTOR_ID, "--stream",
     "What are my dining preferences? Answer in one sentence."],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

In [ ]:
# Give long-term memory extraction a little time, then open a brand-new session
print("Waiting ~30s for long-term memory extraction...")
time.sleep(30)

SESSION_B = f"restaurant-demo-{uuid_module.uuid4()}"
print("SESSION_B (brand new):", SESSION_B)

subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_B, "--actor-id", ACTOR_ID, "--stream",
     "What are my dining preferences? Answer in one sentence using the stored memory."],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

**Point to make:** sessions A and B are completely different conversations -- no shared
history was passed in. The platform retrieved the same stored preferences for the same guest
(`ACTOR_ID`) and injected them before the model saw the prompt. No code change, no custom
DynamoDB table -- this is a harness configuration change, full stop.

## Step 3 — an inline tool (client-side)

Not every tool should run inside the harness. A booking **request** is exactly the kind of
side effect you often want your own application to own -- so it can add human approval,
write to your real reservations system, or apply business rules that have nothing to do with
the LLM.

An **inline function tool** lets the harness describe the tool (name + JSON schema) without
executing it: when the model decides to call it, the harness pauses and returns control to
your application, which runs the tool and (optionally) sends the result back.

We reuse the exact same booking fields already used by `resources/lambdas/restaurant/restaurant-api.json`
elsewhere in this workshop, for consistency.

In [ ]:
tool = next((t for t in harness_config["tools"] if t.get("name") == "request_booking"), None)
if tool is None:
    tool = {"type": "inline_function", "name": "request_booking"}
    harness_config["tools"].append(tool)

tool["type"] = "inline_function"

### Your turn: describe the `request_booking` tool

Set `tool["config"]` to an `inlineFunction` with a `description` and a JSON-Schema
`inputSchema` for `date`, `hour`, `restaurant_name`, `guest_name`, `num_guests` — reuse the
same fields already defined in `resources/lambdas/restaurant/restaurant-api.json`
(`date`/`hour` as strings, `num_guests` as an integer, `date` must reject relative values like
"tomorrow"). Mark `date`, `hour`, `guest_name`, `num_guests` as `required`.

<details>
<summary>Click here for the solution</summary>

```python
tool["config"] = {
    "inlineFunction": {
        "description": "Request a table reservation at The Bedrock Bistro. Does not confirm availability by itself; the client application handles confirmation.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "description": "The date of the booking in the format YYYY-MM-DD. Do NOT accept relative dates like today or tomorrow; ask for the explicit date."},
                "hour": {"type": "string", "description": "The hour of the booking in the format HH:MM."},
                "restaurant_name": {"type": "string", "description": "Name of the restaurant handling the reservation. Defaults to The Bedrock Bistro."},
                "guest_name": {"type": "string", "description": "The name of the guest to have in the reservation."},
                "num_guests": {"type": "integer", "description": "The number of guests for the booking."},
            },
            "required": ["date", "hour", "guest_name", "num_guests"],
        },
    }
}

with open(harness_json_path, "w") as f:
    json.dump(harness_config, f, indent=2)
    f.write("\n")

subprocess.run(["agentcore", "deploy"], cwd=PROJECT_NAME, check=True, env=deploy_env)
```

</details>

In [ ]:
SESSION_3 = f"restaurant-demo-{uuid_module.uuid4()}"
subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_3, "--actor-id", ACTOR_ID, "--stream",
     "I'd like to request a table for 4 on 2026-09-25 at 19:30, under the name Luca."],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

When the harness emits an inline tool call, *your application* is the thing that actually
executes it. Write the client-side executor: a `request_booking(date, hour, guest_name,
num_guests, restaurant_name="The Bedrock Bistro")` function that

- raises `ValueError` if `date`, `hour`, `guest_name`, or `num_guests` is missing,
- builds a `record` dict with a generated `id` (e.g. `f"booking_{uuid.uuid4()}"`), a UTC
  `created_at` timestamp, and the booking fields (`status="requested"`),
- appends that record as one JSON line to `resources/data/booking_requests.jsonl` (create the
  `resources/data` directory if it doesn't exist),
- returns `{"ok": True, "tool": "request_booking", "message": ..., "record": record}`.

<details>
<summary>Click here for the solution</summary>

```python
import datetime
import uuid as uuid_lib

def request_booking(date, hour, guest_name, num_guests, restaurant_name="The Bedrock Bistro"):
    if not (date and hour and guest_name and num_guests):
        raise ValueError("request_booking requires date, hour, guest_name, and num_guests")

    record = {
        "id": f"booking_{uuid_lib.uuid4()}",
        "created_at": datetime.datetime.utcnow().isoformat() + "Z",
        "restaurant_name": restaurant_name,
        "date": date,
        "hour": hour,
        "guest_name": guest_name,
        "num_guests": int(num_guests),
        "status": "requested",
    }

    os.makedirs("resources/data", exist_ok=True)
    with open("resources/data/booking_requests.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")

    return {"ok": True, "tool": "request_booking", "message": f"Booking requested with id {record['id']}", "record": record}
```

</details>

In [ ]:
result = request_booking(date="2026-09-25", hour="19:30", guest_name="Luca", num_guests=4)
print(json.dumps(result, indent=2))

**Point to make:** inline tools are client-side. The harness can pause and hand control
back to your application for human-in-the-loop review, deterministic side effects, or calls
to internal systems the harness itself should never touch directly.

## Step 4 — Gateway: official restaurant data

Right now the model can only talk about the restaurant in general terms, or invent details.
**Gateway** lets us expose the `restaurant_data` Lambda -- already built and zipped at
`resources/lambdas/restaurant_data/` -- as a proper tool the model can call for facts, using the same
`create_gateway_lambda` helper notebook 6 already uses for `calc/` and `restaurant/`.

In [ ]:
restaurant_data_lambda = utils.create_gateway_lambda(
    "resources/lambdas/restaurant_data/lambda_function_code.zip",
    lambda_function_name=f"restaurant_data_lambda_{UUID}",
)
print(restaurant_data_lambda)
RESTAURANT_DATA_LAMBDA_ARN = restaurant_data_lambda["lambda_function_arn"]

In [ ]:
import shutil

os.makedirs(os.path.join(PROJECT_NAME, "tool-schemas"), exist_ok=True)
shutil.copyfile(
    "resources/lambdas/restaurant_data/schema.json",
    os.path.join(PROJECT_NAME, "tool-schemas", "restaurant-data-tools.json"),
)

gateway_name = f"{HARNESS_NAME}-gateway"

subprocess.run(
    ["agentcore", "add", "gateway", "--name", gateway_name,
     "--description", "Gateway for official restaurant data", "--authorizer-type", "AWS_IAM"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

subprocess.run(
    ["agentcore", "add", "gateway-target",
     "--gateway", gateway_name, "--name", "restaurant-data", "--type", "lambda-function-arn",
     "--lambda-arn", RESTAURANT_DATA_LAMBDA_ARN,
     "--tool-schema-file", "tool-schemas/restaurant-data-tools.json"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

subprocess.run(["agentcore", "deploy"], cwd=PROJECT_NAME, check=True, env=deploy_env)

In [ ]:
subprocess.run(
    ["agentcore", "add", "tool",
     "--harness", HARNESS_NAME, "--type", "agentcore_gateway",
     "--name", "restaurant_gateway_data", "--gateway", gateway_name, "--outbound-auth", "awsIam"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

subprocess.run(["agentcore", "deploy"], cwd=PROJECT_NAME, check=True, env=deploy_env)

In [ ]:
SESSION_4 = f"restaurant-demo-{uuid_module.uuid4()}"
subprocess.run(
    ["agentcore", "invoke", "--session-id", SESSION_4, "--actor-id", ACTOR_ID, "--stream",
     "What's on tonight's specials, and what time do you open on Saturday?"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

**Point to make:** Gateway moves restaurant facts behind a controlled tool boundary, so
the model *asks the tool* instead of guessing menu items or opening hours -- directly closing
the gap flagged at the end of Step 1.

## Step 5 — invoke-time overrides (no redeploy)

A few `agentcore invoke` flags let you change behavior **per invocation**, with no code change
and no redeploy: useful for A/B-testing a persona, or debugging.

In [ ]:
# Override the system prompt just for this one call
subprocess.run(
    ["agentcore", "invoke", "--harness", HARNESS_NAME,
     "--system-prompt", "You are a terse restaurant assistant. Always answer in bullet points, in English.",
     "--stream", "Who are you and what can you help me with?"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

In [ ]:
# Verbose mode -- shows every event: tool calls, stop reasons, model chunks
subprocess.run(
    ["agentcore", "invoke", "--harness", HARNESS_NAME, "--verbose", "--stream", "Hi, who are you?"],
    cwd=PROJECT_NAME, check=True, env=deploy_env,
)

Other invoke-time overrides worth knowing about: `--model-id`, `--tools`, `--skills`,
`--max-iterations`, and `--json` (structured output for scripting/automation).

## Troubleshooting

- **Session IDs must be at least 33 characters.** A bare `uuid4()` hex string is 32 characters
  -- just under the limit. This notebook always prefixes it (`f"restaurant-demo-{uuid4()}"`).
- **Memory is keyed on `--actor-id`, not `--session-id`.** Two different sessions with the same
  actor ID share long-term memory; the same session with two different actor IDs does not.
- **The harness's auto-created default execution role lacks Memory permissions.** If you skip
  wiring in a real `executionRoleArn` (as we did in Setup) before adding memory, `agentcore add
  memory` + redeploy will succeed, but memory reads/writes will fail with an access-denied
  error at invoke time -- the fix is exactly what we did above: deploy a role with
  `bedrock-agentcore:*` permissions and set it in `harness.json` before adding memory.

## Next steps

- **Notebook 5** builds a *second*, narrowly-scoped harness and calls it as a tool from this
  one -- the harness-native way to do multi-agent collaboration, no Strands, no custom
  orchestration framework.
- **Notebook 6** adds Skills, Code Interpreter, a Policy guardrail, an Evaluations run, and
  Observability (traces/logs) on top of this same harness.